In [1]:
%pip install -q pgserver psycopg2-binary pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import pgserver
import psycopg2

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

pg = pgserver.get_server('/tmp/fp_pandemic_pg', cleanup_mode='stop')
conn = psycopg2.connect(pg.get_uri())
conn.autocommit = True


def sql(script: str):
    """Виконує SQL-скрипт (один або кілька statement) і повертає результат
    останнього statement як DataFrame, якщо він повертає рядки."""
    with conn.cursor() as cur:
        cur.execute(script)
        if cur.description:
            return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])
        return None


sql('SELECT version()')

,version
0,"PostgreSQL 16.2 on x86_64-pc-linux-gnu, compil..."


In [3]:
sql("""
CREATE SCHEMA IF NOT EXISTS pandemic;
SET search_path TO pandemic;
SHOW search_path;
""")

,search_path
0,pandemic


In [4]:
sql("""
DROP TABLE IF EXISTS infectious_cases_normalized;
DROP TABLE IF EXISTS entities;
DROP TABLE IF EXISTS infectious_cases;

-- Сирі дані імпортуються «як є»: усі колонки TEXT, бо в CSV
-- відсутні значення записані порожніми рядками. Типізація — на етапі нормалізації.
CREATE TABLE infectious_cases (
    entity                TEXT,
    code                  TEXT,
    year                  TEXT,
    number_yaws           TEXT,
    polio_cases           TEXT,
    cases_guinea_worm     TEXT,
    number_rabies         TEXT,
    number_malaria        TEXT,
    number_hiv            TEXT,
    number_tuberculosis   TEXT,
    number_smallpox       TEXT,
    number_cholera_cases  TEXT
);
""")

In [5]:
CSV_CANDIDATES = ['data/infectious_cases.csv', 'infectious_cases.csv', '/content/infectious_cases.csv']
csv_path = next((p for p in CSV_CANDIDATES if os.path.exists(p)), None)

if csv_path is None:
    try:
        from google.colab import files  # у Colab — діалог завантаження файла
        uploaded = files.upload()
        csv_path = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError('Покладіть infectious_cases.csv у data/ або поруч із notebook')

with conn.cursor() as cur, open(csv_path, encoding='utf-8') as f:
    cur.copy_expert("COPY pandemic.infectious_cases FROM STDIN WITH (FORMAT csv, HEADER true)", f)

print('Imported from:', csv_path)
sql('SELECT COUNT(*) AS imported_rows FROM infectious_cases')

Imported from: data/infectious_cases.csv


,imported_rows
0,10521


In [6]:
sql("""
SELECT *
FROM infectious_cases
LIMIT 10;
""")

,entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,Afghanistan,AFG,1980,None,6160,0,None,None,None,None,None,None
1,Afghanistan,AFG,1981,None,5859,0,None,None,None,None,None,None
2,Afghanistan,AFG,1982,None,9730,0,None,None,None,None,None,None
3,Afghanistan,AFG,1983,None,13937,0,None,None,None,None,None,None
4,Afghanistan,AFG,1984,None,3864,0,None,None,None,None,None,None
5,Afghanistan,AFG,1985,None,13867,0,None,None,None,None,None,None
6,Afghanistan,AFG,1986,None,12901,0,None,None,None,None,None,None
7,Afghanistan,AFG,1987,None,4396,0,None,None,None,None,None,None
8,Afghanistan,AFG,1988,None,2149,0,None,None,None,None,None,None
9,Afghanistan,AFG,1989,None,385,0,None,None,None,None,None,None


In [7]:
sql("""
SELECT
    COUNT(*)                                   AS n_rows,
    COUNT(DISTINCT entity)                     AS n_entities,
    COUNT(DISTINCT (entity, code))             AS n_entity_code_pairs,
    COUNT(*) FILTER (WHERE code IS NULL OR TRIM(code) = '') AS rows_without_code,
    MIN(year::INTEGER)                         AS min_year,
    MAX(year::INTEGER)                         AS max_year
FROM infectious_cases;
""")

,n_rows,n_entities,n_entity_code_pairs,rows_without_code,min_year,max_year
0,10521,245,245,1128,1920,2023


In [8]:
sql("""
SELECT
    COUNT(NULLIF(TRIM(number_yaws), '')) AS number_yaws,
    COUNT(NULLIF(TRIM(polio_cases), '')) AS polio_cases,
    COUNT(NULLIF(TRIM(cases_guinea_worm), '')) AS cases_guinea_worm,
    COUNT(NULLIF(TRIM(number_rabies), '')) AS number_rabies,
    COUNT(NULLIF(TRIM(number_malaria), '')) AS number_malaria,
    COUNT(NULLIF(TRIM(number_hiv), '')) AS number_hiv,
    COUNT(NULLIF(TRIM(number_tuberculosis), '')) AS number_tuberculosis,
    COUNT(NULLIF(TRIM(number_smallpox), '')) AS number_smallpox,
    COUNT(NULLIF(TRIM(number_cholera_cases), '')) AS number_cholera_cases
FROM infectious_cases;
""")

,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,74,7610,8428,6780,6780,6780,6780,104,2967


In [9]:
sql("""
-- Допоміжна функція: безпечне перетворення тексту на NUMERIC.
-- Порожні або нечислові значення -> NULL (а не помилка або 0).
CREATE OR REPLACE FUNCTION try_numeric(p_value TEXT)
RETURNS NUMERIC
LANGUAGE sql
IMMUTABLE
AS $$
    SELECT CASE
        WHEN p_value ~ '^\s*-?[0-9]+(\.[0-9]+)?([eE][-+]?[0-9]+)?\s*$' THEN TRIM(p_value)::NUMERIC
    END
$$;
""")

In [10]:
sql("""
DROP TABLE IF EXISTS infectious_cases_normalized;
DROP TABLE IF EXISTS entities;

-- Довідник унікальних комбінацій Entity + Code
CREATE TABLE entities (
    id      SERIAL PRIMARY KEY,
    entity  TEXT NOT NULL,
    code    TEXT,
    CONSTRAINT uq_entities_entity_code UNIQUE NULLS NOT DISTINCT (entity, code)
);

INSERT INTO entities (entity, code)
SELECT DISTINCT
    TRIM(entity),
    NULLIF(TRIM(code), '')
FROM infectious_cases
ORDER BY 1, 2;

SELECT *
FROM entities
ORDER BY id
LIMIT 10;
""")

,id,entity,code
0,1,Afghanistan,AFG
1,2,Africa,NaN
2,3,African Region (WHO),NaN
3,4,Albania,ALB
4,5,Algeria,DZA
5,6,American Samoa,ASM
6,7,Americas,NaN
7,8,Andorra,AND
8,9,Angola,AGO
9,10,Antigua and Barbuda,ATG


In [11]:
sql("""
-- Таблиця показників: замість Entity/Code — зовнішній ключ entity_id
CREATE TABLE infectious_cases_normalized (
    id                    SERIAL PRIMARY KEY,
    entity_id             INTEGER NOT NULL REFERENCES entities (id),
    year                  INTEGER NOT NULL,
    number_yaws           NUMERIC,
    polio_cases           NUMERIC,
    cases_guinea_worm     NUMERIC,
    number_rabies         NUMERIC,
    number_malaria        NUMERIC,
    number_hiv            NUMERIC,
    number_tuberculosis   NUMERIC,
    number_smallpox       NUMERIC,
    number_cholera_cases  NUMERIC,
    CONSTRAINT uq_cases_entity_year UNIQUE (entity_id, year)
);

INSERT INTO infectious_cases_normalized (
    entity_id, year,
    number_yaws, polio_cases, cases_guinea_worm, number_rabies, number_malaria,
    number_hiv, number_tuberculosis, number_smallpox, number_cholera_cases
)
SELECT
    e.id,
    ic.year::INTEGER,
    try_numeric(ic.number_yaws),
    try_numeric(ic.polio_cases),
    try_numeric(ic.cases_guinea_worm),
    try_numeric(ic.number_rabies),
    try_numeric(ic.number_malaria),
    try_numeric(ic.number_hiv),
    try_numeric(ic.number_tuberculosis),
    try_numeric(ic.number_smallpox),
    try_numeric(ic.number_cholera_cases)
FROM infectious_cases AS ic
JOIN entities AS e
  ON e.entity = TRIM(ic.entity)
 AND e.code IS NOT DISTINCT FROM NULLIF(TRIM(ic.code), '')
ORDER BY e.id, ic.year::INTEGER;

SELECT *
FROM infectious_cases_normalized
ORDER BY id
LIMIT 10;
""")

,id,entity_id,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,1,1,1960,None,None,None,None,None,None,None,None,887
1,2,1,1965,None,None,None,None,None,None,None,None,218
2,3,1,1980,None,6160,0,None,None,None,None,None,None
3,4,1,1981,None,5859,0,None,None,None,None,None,None
4,5,1,1982,None,9730,0,None,None,None,None,None,None
5,6,1,1983,None,13937,0,None,None,None,None,None,None
6,7,1,1984,None,3864,0,None,None,None,None,None,None
7,8,1,1985,None,13867,0,None,None,None,None,None,None
8,9,1,1986,None,12901,0,None,None,None,None,None,None
9,10,1,1987,None,4396,0,None,None,None,None,None,None


In [12]:
sql("""
SELECT
    table_name,
    ordinal_position AS pos,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'pandemic'
  AND table_name IN ('entities', 'infectious_cases_normalized')
ORDER BY table_name, ordinal_position;
""")

,table_name,pos,column_name,data_type,is_nullable
0,entities,1,id,integer,NO
1,entities,2,entity,text,NO
2,entities,3,code,text,YES
3,infectious_cases_normalized,1,id,integer,NO
4,infectious_cases_normalized,2,entity_id,integer,NO
5,infectious_cases_normalized,3,year,integer,NO
6,infectious_cases_normalized,4,number_yaws,numeric,YES
7,infectious_cases_normalized,5,polio_cases,numeric,YES
8,infectious_cases_normalized,6,cases_guinea_worm,numeric,YES
9,infectious_cases_normalized,7,number_rabies,numeric,YES


In [13]:
sql("""
-- Перевірка без втрат: кількість рядків збігається,
-- а JOIN нормалізованих таблиць відтворює оригінал (EXCEPT = 0 рядків)
SELECT
    (SELECT COUNT(*) FROM infectious_cases)            AS raw_rows,
    (SELECT COUNT(*) FROM infectious_cases_normalized) AS normalized_rows,
    (SELECT COUNT(*) FROM entities)                    AS entity_rows,
    (
        SELECT COUNT(*) FROM (
            SELECT TRIM(entity), NULLIF(TRIM(code), ''), year::INTEGER,
                   try_numeric(number_rabies), try_numeric(number_malaria)
            FROM infectious_cases
            EXCEPT
            SELECT e.entity, e.code, n.year, n.number_rabies, n.number_malaria
            FROM infectious_cases_normalized AS n
            JOIN entities AS e ON e.id = n.entity_id
        ) AS diff
    ) AS rows_lost_or_changed;
""")

,raw_rows,normalized_rows,entity_rows,rows_lost_or_changed
0,10521,10521,245,0


In [14]:
sql("""
SELECT
    e.id                              AS entity_id,
    e.entity,
    e.code,
    ROUND(AVG(n.number_rabies), 4)    AS avg_rabies,
    MIN(n.number_rabies)              AS min_rabies,
    MAX(n.number_rabies)              AS max_rabies,
    ROUND(SUM(n.number_rabies), 4)    AS sum_rabies
FROM infectious_cases_normalized AS n
JOIN entities AS e
  ON e.id = n.entity_id
WHERE n.number_rabies IS NOT NULL
GROUP BY e.id, e.entity, e.code
ORDER BY AVG(n.number_rabies) DESC
LIMIT 10;
""")

,entity_id,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies
0,241,World,OWID_WRL,20192.3704,14075.508,24744.658,605771.1110
1,121,Lower Middle Income (WB),NaN,15193.9594,10202.528,19182.795,455818.7815
2,202,South Asia (WB),NaN,11729.8888,7271.2754,15361.878,351896.6626
3,206,South-East Asia Region (WHO),NaN,11424.3269,6806.0127,15641.958,342729.8068
4,77,G20,NaN,10189.0460,6339.0796,13164.881,305671.3801
5,97,India,IND,8599.1733,5425.8726,11121.142,257975.1984
6,209,Sub-Saharan Africa (WB),NaN,4748.2681,4356.8184,5017.977,142448.0427
7,3,African Region (WHO),NaN,4649.7024,4283.073,4917.784,139491.0713
8,120,Low Income (WB),NaN,3568.1055,3014.617,3873.533,107043.1635
9,58,East Asia & Pacific (WB),NaN,3427.4611,2009.9395,4591.0107,102823.8340


In [15]:
sql("""
SELECT
    n.id,
    e.entity,
    e.code,
    n.year,
    MAKE_DATE(n.year, 1, 1)                                         AS year_start_date,
    CURRENT_DATE                                                    AS today,
    EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(n.year, 1, 1)))::INTEGER AS years_diff
FROM infectious_cases_normalized AS n
JOIN entities AS e
  ON e.id = n.entity_id
ORDER BY n.id
LIMIT 10;
""")

,id,entity,code,year,year_start_date,today,years_diff
0,1,Afghanistan,AFG,1960,1960-01-01,2026-10-18,66
1,2,Afghanistan,AFG,1965,1965-01-01,2026-10-18,61
2,3,Afghanistan,AFG,1980,1980-01-01,2026-10-18,46
3,4,Afghanistan,AFG,1981,1981-01-01,2026-10-18,45
4,5,Afghanistan,AFG,1982,1982-01-01,2026-10-18,44
5,6,Afghanistan,AFG,1983,1983-01-01,2026-10-18,43
6,7,Afghanistan,AFG,1984,1984-01-01,2026-10-18,42
7,8,Afghanistan,AFG,1985,1985-01-01,2026-10-18,41
8,9,Afghanistan,AFG,1986,1986-01-01,2026-10-18,40
9,10,Afghanistan,AFG,1987,1987-01-01,2026-10-18,39


In [16]:
sql("""
-- Функція 1: рік -> різниця в повних роках між CURRENT_DATE та 1 січня цього року
CREATE OR REPLACE FUNCTION fn_years_since(p_year INTEGER)
RETURNS INTEGER
LANGUAGE sql
STABLE                       -- залежить від CURRENT_DATE, тому не IMMUTABLE
AS $$
    SELECT EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1)))::INTEGER
$$;

SELECT
    1996                  AS input_year,
    MAKE_DATE(1996, 1, 1) AS year_start_date,
    CURRENT_DATE          AS today,
    fn_years_since(1996)  AS years_diff;
""")

,input_year,year_start_date,today,years_diff
0,1996,1996-01-01,2026-10-18,30


In [17]:
sql("""
SELECT
    n.id,
    e.entity,
    n.year,
    MAKE_DATE(n.year, 1, 1)                                         AS year_start_date,
    CURRENT_DATE                                                    AS today,
    fn_years_since(n.year)                                          AS years_diff_fn,
    EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(n.year, 1, 1)))::INTEGER AS years_diff_builtin
FROM infectious_cases_normalized AS n
JOIN entities AS e
  ON e.id = n.entity_id
ORDER BY n.id
LIMIT 10;
""")

,id,entity,year,year_start_date,today,years_diff_fn,years_diff_builtin
0,1,Afghanistan,1960,1960-01-01,2026-10-18,66,66
1,2,Afghanistan,1965,1965-01-01,2026-10-18,61,61
2,3,Afghanistan,1980,1980-01-01,2026-10-18,46,46
3,4,Afghanistan,1981,1981-01-01,2026-10-18,45,45
4,5,Afghanistan,1982,1982-01-01,2026-10-18,44,44
5,6,Afghanistan,1983,1983-01-01,2026-10-18,43,43
6,7,Afghanistan,1984,1984-01-01,2026-10-18,42,42
7,8,Afghanistan,1985,1985-01-01,2026-10-18,41,41
8,9,Afghanistan,1986,1986-01-01,2026-10-18,40,40
9,10,Afghanistan,1987,1987-01-01,2026-10-18,39,39


In [18]:
sql("""
-- Функція дає той самий результат, що й вбудовані функції з Завдання 4, для всіх рядків
SELECT
    COUNT(*) AS n_rows_checked,
    COUNT(*) FILTER (
        WHERE fn_years_since(year)
              <> EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1)))::INTEGER
    ) AS n_mismatches
FROM infectious_cases_normalized;
""")

,n_rows_checked,n_mismatches
0,10521,0


In [19]:
sql("""
-- Функція 2: кількість захворювань за рік / довільний дільник
CREATE OR REPLACE FUNCTION fn_avg_cases_per_period(
    p_cases_per_year NUMERIC,
    p_divider        NUMERIC
)
RETURNS NUMERIC
LANGUAGE plpgsql
IMMUTABLE
AS $$
BEGIN
    IF p_divider IS NULL OR p_divider = 0 THEN
        RAISE EXCEPTION 'fn_avg_cases_per_period: divider must be non-zero (got %)', p_divider;
    END IF;
    RETURN p_cases_per_year / p_divider;
END;
$$;

SELECT
    ROUND(fn_avg_cases_per_period(1200, 12), 2) AS per_month_1200,
    ROUND(fn_avg_cases_per_period(1200, 4), 2)  AS per_quarter_1200,
    ROUND(fn_avg_cases_per_period(1200, 2), 2)  AS per_half_year_1200;
""")

,per_month_1200,per_quarter_1200,per_half_year_1200
0,100.00,300.00,600.00


In [20]:
sql("""
-- Скільки записів Number_rabies мають числове значення (лише їх передаємо у функцію)
SELECT
    COUNT(*)                                                    AS total_rows,
    COUNT(*) FILTER (WHERE number_rabies ~ '^\s*-?[0-9]+(\.[0-9]+)?([eE][-+]?[0-9]+)?\s*$')     AS numeric_rows,
    COUNT(*) FILTER (WHERE number_rabies IS NULL
                        OR number_rabies !~ '^\s*-?[0-9]+(\.[0-9]+)?([eE][-+]?[0-9]+)?\s*$')    AS non_numeric_or_empty_rows
FROM infectious_cases;
""")

,total_rows,numeric_rows,non_numeric_or_empty_rows
0,10521,6780,3741


In [21]:
sql("""
SELECT
    entity,
    code,
    year::INTEGER                                                       AS year,
    number_rabies::NUMERIC                                              AS rabies_per_year,
    ROUND(fn_avg_cases_per_period(number_rabies::NUMERIC, 12), 4)       AS avg_per_month,
    ROUND(fn_avg_cases_per_period(number_rabies::NUMERIC, 4), 4)        AS avg_per_quarter,
    ROUND(fn_avg_cases_per_period(number_rabies::NUMERIC, 2), 4)        AS avg_per_half_year
FROM infectious_cases
WHERE number_rabies ~ '^\s*-?[0-9]+(\.[0-9]+)?([eE][-+]?[0-9]+)?\s*$'       -- лише числові значення
ORDER BY entity, year::INTEGER
LIMIT 10;
""")

,entity,code,year,rabies_per_year,avg_per_month,avg_per_quarter,avg_per_half_year
0,Afghanistan,AFG,1990,2.145772,0.1788,0.5364,1.0729
1,Afghanistan,AFG,1991,2.3209748,0.1934,0.5802,1.1605
2,Afghanistan,AFG,1992,2.6180408,0.2182,0.6545,1.3090
3,Afghanistan,AFG,1993,2.8531172,0.2378,0.7133,1.4266
4,Afghanistan,AFG,1994,2.9899871,0.2492,0.7475,1.4950
5,Afghanistan,AFG,1995,3.126361,0.2605,0.7816,1.5632
6,Afghanistan,AFG,1996,3.281437,0.2735,0.8204,1.6407
7,Afghanistan,AFG,1997,3.446122,0.2872,0.8615,1.7231
8,Afghanistan,AFG,1998,3.5570247,0.2964,0.8893,1.7785
9,Afghanistan,AFG,1999,3.607037,0.3006,0.9018,1.8035


## Висновки

- Схему `pandemic` створено і встановлено за замовчуванням через `SET search_path`. CSV імпортовано через `COPY` у staging-таблицю `infectious_cases`.
- **3НФ:** `entities(id, entity, code)` зберігає кожну пару Entity+Code один раз, а `infectious_cases_normalized` посилається на неї через `entity_id`. Неключові атрибути залежать лише від ключа `(entity_id, year)`, і транзитивної залежності `code ← entity` у таблиці показників немає. Перевірка `EXCEPT` підтвердила, що дані відтворюються без втрат.
- Агрегати `Number_rabies` рахуються тільки за числовими значеннями (`NULL` не враховуються). Результат відсортовано за середнім за спаданням і обмежено 10 рядками.
- Різниця в роках рахується вбудованими `MAKE_DATE` + `CURRENT_DATE` + `AGE`. Функція `fn_years_since` дає той самий результат для всіх рядків (0 розбіжностей).
- `fn_avg_cases_per_period` застосовано лише до числових значень з дільниками 12 / 4 / 2.